# 08 — Predicting the 2026 turnout change

**Owner:** F + A  
**Goal:** a model that beats the baseline, with per-municipality explanations and prediction intervals

**Reads:** `PROCESSED / "master.csv"`  
**Writes:** `PROCESSED / "predictions_2026.csv"`

Run the two setup cells below first, every time.

## Setup

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        print("No GITHUB_TOKEN in Colab Secrets - cloning will fail for the private repo")
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

config.make_dirs(); config.set_seed()
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
print(config.describe())

## Target and split

Target = CHANGE in turnout, never the level. Train 2011→2016, test 2016→2021. Never a random split.

In [ ]:
# y = master["t"] - master["t_prev"]
# train = master[master["election"] == 2016]
# test  = master[master["election"] == 2021]

## Baseline

2026 turnout = 2021 turnout. Everything must beat this.

In [ ]:
from sklearn.metrics import mean_absolute_error as mae
# mae(test["t"], test["t_prev"])

## Ridge

Interpretable coefficients; scale the features first.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## Hierarchical model

Municipalities grouped within provinces — partial pooling for a small sample.

In [ ]:
import statsmodels.formula.api as smf
# m = smf.mixedlm("dt ~ ...", data=train, groups=train["province"]).fit()

## LightGBM + SHAP

Non-linear patterns, then explanations per municipality.

In [ ]:
import lightgbm as lgb, shap

## Results table

MAE and RMSE for all four, side by side with the baseline. Report it honestly.

## 2026 forecast

p_2026 = r_2026 × predicted t_2026, plus a prediction interval.

## Worse than expected

Largest negative residuals — local factors our data cannot see.

## Save

Model and predictions, for the dashboard.

In [ ]:
import joblib
# joblib.dump(model, MODELS / "turnout_model.joblib")

## Save output

In [ ]:
# out.to_csv(PROCESSED / "predictions_2026.csv", index=False)
# print(out.shape); out.head()